# Librerias

In [ ]:
%pip install pandas numpy scikit-learn joblib matplotlib streamlit openpyxl pyngrok

In [ ]:
# === Seccion 1: Importar librerias y resolver rutas del proyecto ===
from pathlib import Path
import os
import threading

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
 )
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    cross_validate,
    train_test_split,
 )

# Dependencias opcionales de infraestructura (no bloquean flujo analitico local)
try:
    from google.colab import drive  # type: ignore
except Exception:
    drive = None

try:
    from pyngrok import ngrok  # type: ignore
except Exception:
    ngrok = None

def resolve_project_root() -> Path:
    """Encuentra la raiz del proyecto para soportar ejecucion desde raiz o /notebooks."""
    candidates = [Path.cwd(), Path.cwd().parent]
    for base in candidates:
        if (base / "data" / "mental_health_digital_behavior_data.csv").exists():
            return base
    # Fallback: permite seguir ejecutando y mostrar error claro al cargar datos
    return Path.cwd()

PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PRIVATE_DATA_DIR = PROJECT_ROOT / "private-data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

print(f"PROJECT_ROOT resuelto en: {PROJECT_ROOT}")

# Cargar Base de entrenamiento

In [ ]:
file_path = DATA_DIR / "mental_health_digital_behavior_data.csv"

try:
    df = pd.read_csv(file_path, on_bad_lines="skip")
    print("Archivo cargado exitosamente:", file_path)
    print("Numero de filas y columnas:", df.shape)
    display(df.head())
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontro el dataset canonico en {file_path}. "
        "Verifica la estructura del repositorio y la resolucion de PROJECT_ROOT."
    )

# Cargar Base del formulario

In [ ]:
# Ruta local del formulario (dataset privado)
ruta_local = PRIVATE_DATA_DIR / "form_responses.csv"

if ruta_local.exists():
    df_formulario = pd.read_csv(ruta_local)
    print(f"Formulario privado cargado: {ruta_local}")
else:
    # Ajuste de infraestructura: fallback no sensible para permitir ejecucion reproducible
    # del flujo analitico cuando no hay acceso al archivo privado.
    df_formulario = df.copy()
    print(
        "Formulario privado no disponible. "
        "Se usa temporalmente una copia de df para mantener ejecutable el flujo completo."
    )

df_formulario.head()

# Seccion 1: Clasificador de Riesgo

## Crear variable objetivo

In [ ]:
# === Seccion 3: Crear la variable 'at_risk' segun la formula definida ===
# Ajuste de compatibilidad de tipos: conserva la misma conversion decimal para entradas string
df['anxiety_level'] = pd.to_numeric(
    df['anxiety_level'].astype(str).str.replace(',', '.', regex=False), errors='coerce'
)
df['mood_score'] = pd.to_numeric(
    df['mood_score'].astype(str).str.replace(',', '.', regex=False), errors='coerce'
)

df['at_risk'] = ((df['anxiety_level'] >= 8) | (df['mood_score'] <= 6)).astype(int)

## Separación de features y target

In [ ]:
# === Sección 4: Definir variables predictoras (X) y variable objetivo (y) ===

# Eliminar columnas no necesarias si las hay, o seleccionar solo las relevantes
X = df.drop(columns=['at_risk'])  # automáticamente toma todas las columnas excepto la de destino
y = df['at_risk']

# En caso de que haya variables categóricas, convertirlas a numéricas (si aplica)
X = pd.get_dummies(X, drop_first=True)  # en caso de que existan columnas categóricas


## Definición del modelo y validación cruzada (10 folds)

In [ ]:
# === Sección 5: Evaluación con validación cruzada y matriz de confusión ===

# Define the model
model = LogisticRegression()

# Predicciones de validación cruzada (imitando Orange)
y_pred_cv = cross_val_predict(model, X, y, cv=10)

# Calcular matriz de confusión
cm_cv = confusion_matrix(y, y_pred_cv)

# Mostrar la matriz como tabla
print("\nConfusion Matrix (Cross-validation):")
print(pd.DataFrame(cm_cv,
                   index=["Real: 0 (No riesgo)", "Real: 1 (Riesgo)"],
                   columns=["Predicho: 0", "Predicho: 1"]))

# Mostrar visualmente
disp = ConfusionMatrixDisplay(confusion_matrix=cm_cv,
                              display_labels=["No riesgo", "Riesgo"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Cross-validation")
plt.show()

## Guardado del modelo entrenado

In [ ]:
# === SECCION 6: Guardar el modelo entrenado (ruta local reproducible) ===
save_path = MODELS_DIR
os.makedirs(save_path, exist_ok=True)

# Fit del modelo de clasificacion sobre todo el dataset de entrenamiento
model.fit(X, y)

model_filename = "logistic_model_mental_health.joblib"
classification_model_path = save_path / model_filename
joblib.dump(model, classification_model_path)

print(f"Modelo guardado exitosamente en: {classification_model_path}")

# Seccion 2: Prediccion de "Well_digital_being_indicator"

## Eliminar/prevenir el preprocesamiento anterior

In [ ]:
# Eliminar columnas procesadas relacionadas a 'at_risk', si las hay
cols_to_drop = [col for col in df.columns if 'at_risk' in col.lower() and col != 'at_risk']
df.drop(columns=cols_to_drop, inplace=True)

# Si at_risk fue transformada y quieres restaurarla a valores originales (por ejemplo, texto), hazlo aquí:
# Ejemplo si se codificó: 1 = 'At risk', 0 = 'No risk'
if 'at_risk' in df.columns and df['at_risk'].dtype in [int, float]:
    df['at_risk'] = df['at_risk'].map({1: 'At risk', 0: 'No risk'})

# Mostrar columnas actuales
print("Columnas después de limpiar:", df.columns.tolist())


## Calcular la correlación de todas las variables con "Well_digital_being_indicator"

In [ ]:


# Asegurar que los valores con comas se conviertan correctamente a numéricos
for col in ['daily_screen_time_min', 'num_app_switches', 'sleep_hours', 'notification_count', 'social_media_time_min', 'focus_score', 'digital_wellbeing_score']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Filtrar solo columnas numéricas
numeric_df = df.select_dtypes(include=np.number)
correlation_matrix = numeric_df.corr()

# Verificar si existe la columna objetivo
if 'digital_wellbeing_score' in correlation_matrix.columns:
    well_being_corr = correlation_matrix['digital_wellbeing_score'].drop('digital_wellbeing_score', errors='ignore')

    # Crear tabla con el signo de la correlación
    correlation_df = pd.DataFrame({
        'Variable': well_being_corr.index,
        'Correlation': well_being_corr.values
    })

    # Ordenar por valor absoluto, manteniendo el signo
    correlation_df['AbsCorrelation'] = correlation_df['Correlation'].abs()
    correlation_df = correlation_df.sort_values(by='AbsCorrelation', ascending=False).drop(columns='AbsCorrelation').reset_index(drop=True)

    # Mostrar tabla
    display(correlation_df)


## Seleccion de features y entrenamiento del modelo de regresion lineal

In [ ]:

# 1. Seleccionar automáticamente las top 3 features
top_3 = correlation_df['Variable'].iloc[:3].tolist()

# Filtrar datos, eliminando filas con NaN en las variables seleccionadas
df_model = df[top_3 + ['digital_wellbeing_score']].dropna()

# Definir X e y
X = df_model[top_3]
y = df_model['digital_wellbeing_score']

# Separar en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entrenar el modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Predecir
y_pred = model.predict(X_test)

# Calcular métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

# 2. Mostrar coeficientes en tabla
coef_df = pd.DataFrame({
    'Variable': top_3,
    'Coeficiente': model.coef_.round(4)
})
coef_df.loc[len(coef_df)] = ['Intercepto', round(model.intercept_, 4)]
print("📊 Coeficientes del modelo:\n")
display(coef_df)

# 3. Mostrar métricas en tabla
metrics_df = pd.DataFrame({
    'Métrica': ['R²', 'MSE', 'RMSE', 'MAE', 'MAPE (%)'],
    'Valor': [round(v, 4) for v in [r2, mse, rmse, mae, mape]]
})
print("📈 Métricas del modelo en conjunto de prueba:\n")
display(metrics_df)


## Guardado de modelo entrenado

In [ ]:
# Guardado local reproducible del modelo de regresion
save_path = MODELS_DIR
os.makedirs(save_path, exist_ok=True)

model_filename = "linear_regression_digital_wellbeing.joblib"
regression_model_path = save_path / model_filename
joblib.dump(model, regression_model_path)

print(f"Modelo guardado exitosamente en: {regression_model_path}")

# Seccion 3: Aplicacion de los modelos en la base de datos del formulario

# Cargar modelos y predecir en una nueva base de datos real

In [ ]:
# Ruta base de los modelos
modelos_path = MODELS_DIR

# Cargar modelo de riesgo mental (clasificacion)
modelo_riesgo = joblib.load(modelos_path / "logistic_model_mental_health.joblib")

# Cargar modelo de bienestar digital (regresion)
modelo_bienestar = joblib.load(modelos_path / "linear_regression_digital_wellbeing.joblib")

print("Modelos cargados correctamente.")

## Aplicar modelos

In [ ]:
# Flujo reproducible local para aplicar modelos sobre formulario
modelos_path = MODELS_DIR

try:
    modelo_riesgo = joblib.load(modelos_path / "logistic_model_mental_health.joblib")
    print("Modelo de riesgo mental (clasificacion) cargado correctamente.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontro el modelo de riesgo en {modelos_path / 'logistic_model_mental_health.joblib'}"
    )

try:
    modelo_bienestar = joblib.load(modelos_path / "linear_regression_digital_wellbeing.joblib")
    print("Modelo de bienestar digital (regresion) cargado correctamente.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontro el modelo de regresion en {modelos_path / 'linear_regression_digital_wellbeing.joblib'}"
    )

# Cargar el archivo de formulario privado si existe; fallback reproducible si no existe
ruta_local_formulario = PRIVATE_DATA_DIR / "form_responses.csv"
if ruta_local_formulario.exists():
    df_formulario = pd.read_csv(ruta_local_formulario)
    print(f"Archivo '{ruta_local_formulario.name}' cargado exitosamente.")
    print("Numero de filas y columnas del formulario:", df_formulario.shape)
else:
    df_formulario = df.copy()
    print(
        "Archivo de formulario privado no disponible. "
        "Se usa df como fallback para mantener ejecutable la validacion."
    )

print("--- Creando la variable 'at_risk' en df_resultados ---")
df_resultados = df_formulario.copy()

for col in ["anxiety_level", "mood_score"]:
    if col in df_resultados.columns:
        df_resultados[col] = df_resultados[col].astype(str).str.replace(",", ".", regex=False)
        df_resultados[col] = pd.to_numeric(df_resultados[col], errors="coerce")
    else:
        print(f"Advertencia: La columna '{col}' no se encontro en df_resultados.")

initial_rows_at_risk = df_resultados.shape[0]
df_resultados.dropna(subset=["anxiety_level", "mood_score"], inplace=True)
if df_resultados.shape[0] < initial_rows_at_risk:
    print(
        f"Se eliminaron {initial_rows_at_risk - df_resultados.shape[0]} filas con NaN "
        "en 'anxiety_level' o 'mood_score'."
    )

df_resultados["at_risk"] = (
    (df_resultados["anxiety_level"] >= 8) | (df_resultados["mood_score"] <= 6)
).astype(int)

print("Variable 'at_risk' creada en df_resultados.")
display(df_resultados[["anxiety_level", "mood_score", "at_risk"]].head())
print(f"Conteo de 'at_risk':\n{df_resultados['at_risk'].value_counts()}")

print("\n--- Aplicando el modelo de clasificacion para predecir 'at_risk' ---")
X_clasificacion_para_prediccion = df_resultados.copy()
if "at_risk" in X_clasificacion_para_prediccion.columns:
    X_clasificacion_para_prediccion = X_clasificacion_para_prediccion.drop(columns=["at_risk"]

    )

X_clasificacion_para_prediccion = pd.get_dummies(
    X_clasificacion_para_prediccion, drop_first=True
)

if hasattr(modelo_riesgo, "feature_names_in_"):
    training_cols_clasificacion = modelo_riesgo.feature_names_in_
    X_clasificacion_para_prediccion = X_clasificacion_para_prediccion.reindex(
        columns=training_cols_clasificacion, fill_value=0
    )
    print("Columnas de clasificacion alineadas con entrenamiento.")
else:
    print(
        "Advertencia: No se pudo obtener 'feature_names_in_' del modelo de riesgo."
    )

_at_risk_predicho = modelo_riesgo.predict(X_clasificacion_para_prediccion)
_probabilidad_at_risk_predicha = modelo_riesgo.predict_proba(
    X_clasificacion_para_prediccion
)[:, 1]

print("Predicciones de 'at_risk' generadas.")
print(f"Conteo predicho de 'at_risk':\n{pd.Series(_at_risk_predicho).value_counts()}")

print("\n--- Calculando el 'Indice de Bienestar Digital' ---")
features_regresion_nuevas = ["anxiety_level", "sleep_hours", "focus_score"]

for col in features_regresion_nuevas:
    if col in df_resultados.columns:
        df_resultados[col] = df_resultados[col].astype(str).str.replace(",", ".", regex=False)
        df_resultados[col] = pd.to_numeric(df_resultados[col], errors="coerce")
    else:
        print(f"Advertencia: La columna '{col}' no se encontro en df_resultados.")

df_regresion_para_prediccion = df_resultados.dropna(subset=features_regresion_nuevas).copy()

if df_regresion_para_prediccion.empty:
    print("No hay datos suficientes para calcular 'digital_wellbeing_score'.")
    df_resultados["digital_wellbeing_score"] = np.nan
else:
    X_regresion_para_prediccion = df_regresion_para_prediccion[features_regresion_nuevas]
    if hasattr(modelo_bienestar, "feature_names_in_"):
        training_cols_regresion = modelo_bienestar.feature_names_in_
        X_regresion_para_prediccion = X_regresion_para_prediccion.reindex(
            columns=training_cols_regresion, fill_value=0
        )
        print("Columnas de regresion alineadas con entrenamiento.")
    else:
        print(
            "Advertencia: No se pudo obtener 'feature_names_in_' del modelo de bienestar."
        )

    df_regresion_para_prediccion["digital_wellbeing_score"] = modelo_bienestar.predict(
        X_regresion_para_prediccion
    )

    df_resultados = df_resultados.merge(
        df_regresion_para_prediccion[["digital_wellbeing_score"]],
        left_index=True,
        right_index=True,
        how="left",
    )
    print("'digital_wellbeing_score' calculado y agregado a df_resultados.")
    display(
        df_resultados[features_regresion_nuevas + ["digital_wellbeing_score"]].head()
    )
    print("\nEstadisticas descriptivas de 'digital_wellbeing_score':")
    print(df_resultados["digital_wellbeing_score"].describe())

print("\n--- Convirtiendo 'at_risk' a formato legible ---")
df_resultados["at_risk"] = df_resultados["at_risk"].map({1: "Riesgo", 0: "No riesgo"})
print("Columna 'at_risk' actualizada a texto.")

print("\n### Tabla Final 'df_resultados' ###")
display(df_resultados.head())
print(f"\nNumero total de filas y columnas: {df_resultados.shape}")
print("Columnas finales:")
print(df_resultados.columns.tolist())

# Guardar resultados en salida local
save_directory = OUTPUTS_DIR
file_name = "Resultados Finales(Python).xlsx"
full_path = save_directory / file_name
os.makedirs(save_directory, exist_ok=True)
df_resultados.to_excel(full_path, index=False)
print(f"Tabla final guardada en: {full_path}")

## Guardar Resultados

In [ ]:
# Guardado redundante de resultados (ruta local reproducible)
save_directory = OUTPUTS_DIR
file_name = "Resultados Finales(Python).xlsx"
full_path = save_directory / file_name

os.makedirs(save_directory, exist_ok=True)
df_resultados.to_excel(full_path, index=False)
print(f"Tabla final guardada en: {full_path}")

#Aplicacion en streamlit

In [ ]:
# Infraestructura Streamlit separada del flujo analitico
app_code = """
import os

import joblib
import numpy as np
import pandas as pd
import streamlit as st

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(BASE_DIR, ".."))
MODELOS_PATH = os.path.join(PROJECT_ROOT, "models")

try:
    modelo_riesgo = joblib.load(os.path.join(MODELOS_PATH, "logistic_model_mental_health.joblib"))
    modelo_bienestar = joblib.load(
        os.path.join(MODELOS_PATH, "linear_regression_digital_wellbeing.joblib")
    )
except FileNotFoundError as e:
    st.error(f"Error cargando modelos: {e}. Verifica que los archivos existan.")
    st.stop()
except Exception as e:
    st.error(f"Ocurrio un error inesperado cargando modelos: {e}")
    st.stop()

st.title("Evaluacion de Bienestar Digital y Riesgo Mental")
st.header("Introduce tus datos:")

daily_screen_time_min = st.number_input(
    "Cuantos minutos al dia usas el celular o computador?", min_value=0, value=240
)
num_app_switches = st.number_input(
    "Cuantas veces cambias entre aplicaciones en un dia?", min_value=0, value=50
)
social_media_time_min = st.number_input(
    "Cuantos minutos usas redes sociales diariamente?", min_value=0, value=100
)
notification_count = st.number_input(
    "Cuantas notificaciones de redes sociales recibes al dia?", min_value=0, value=150
)
sleep_hours = st.number_input(
    "Cuantas horas duermes por noche?", min_value=0.0, max_value=24.0, value=7.0
)
focus_score = st.slider("Que tan enfocado(a) te sentiste esta semana?", 1, 10, value=7)
anxiety_level = st.slider(
    "Que tan ansioso(a) o estresado(a) te sentiste esta semana?", 1, 10, value=6
)
mood_score = st.slider(
    "Como calificarias tu estado de animo general esta semana?", 1, 10, value=8
)

if st.button("Evaluar"):
    input_data_dict = {
        "daily_screen_time_min": daily_screen_time_min,
        "num_app_switches": num_app_switches,
        "sleep_hours": sleep_hours,
        "notification_count": notification_count,
        "social_media_time_min": social_media_time_min,
        "focus_score": focus_score,
        "mood_score": mood_score,
        "anxiety_level": anxiety_level,
    }

    entrada_clasificacion_df = pd.DataFrame([input_data_dict])

    if hasattr(modelo_riesgo, "feature_names_in_"):
        expected_cols_riesgo = list(modelo_riesgo.feature_names_in_)
        entrada_clasificacion_df = entrada_clasificacion_df.reindex(
            columns=expected_cols_riesgo, fill_value=0
        )
    else:
        st.warning("No se puede verificar el orden de columnas para el modelo de riesgo.")
        if entrada_clasificacion_df.shape[1] != 8:
            st.error(
                f"Numero inesperado de columnas: {entrada_clasificacion_df.shape[1]}. Se esperaban 8."
            )
            st.stop()

    entrada_regresion = np.array([[anxiety_level, sleep_hours, focus_score]])

    riesgo_pred = modelo_riesgo.predict(entrada_clasificacion_df)[0]
    probabilidad_riesgo = modelo_riesgo.predict_proba(entrada_clasificacion_df)[:, 1][0]

    if hasattr(modelo_bienestar, "feature_names_in_"):
        entrada_regresion_df = pd.DataFrame(
            [[anxiety_level, sleep_hours, focus_score]],
            columns=["anxiety_level", "sleep_hours", "focus_score"],
        )
        entrada_regresion_df = entrada_regresion_df.reindex(
            columns=modelo_bienestar.feature_names_in_, fill_value=0
        )
        bienestar = modelo_bienestar.predict(entrada_regresion_df)[0]
    else:
        bienestar = modelo_bienestar.predict(entrada_regresion)[0]

    st.subheader("Resultados de la Evaluacion:")
    riesgo_texto = "Si, hay un riesgo potencial" if riesgo_pred == 1 else "No, el riesgo es bajo"
    st.markdown(f"**Riesgo Mental:** {riesgo_texto}")
    st.info(f"*(Probabilidad de Riesgo: {probabilidad_riesgo:.2%})*")

    st.markdown(f"**Puntaje de Bienestar Digital:** {bienestar:.2f}")
    if bienestar >= 70:
        st.success("Excelente. Tu bienestar digital parece muy saludable.")
    elif bienestar >= 50:
        st.info("Buen trabajo, tu bienestar digital es moderado.")
    else:
        st.warning("Considera revisar tus habitos digitales para mejorar tu bienestar.")

    st.markdown("---")
    st.markdown(
        "Estos resultados son una estimacion y no sustituyen la evaluacion de un profesional."
    )
"""

# Escribir aplicacion de referencia
output_app_path = PROJECT_ROOT / "app" / "app.py"
os.makedirs(output_app_path.parent, exist_ok=True)
with open(output_app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"app.py actualizado en: {output_app_path}")
print("Ejecucion recomendada: streamlit run app/app.py")